<a href="https://colab.research.google.com/github/charukitch68-bit/project-airline-/blob/main/airline_passenger_satisfaction_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project: Airline Passenger Satisfaction
> 1145 201 คณิตศาสตร์สำหรับวิทยาการข้อมูล | Semester 1/2568

**กลุ่ม:** [ระบุชื่อสมาชิกและรหัสนักศึกษา]
- สมาชิกที่ 1: [นายจิรีฐิตกาล ลาเลิศ] ([68114540146])
- สมาชิกที่ 2: [นายจารุกิตติ์ ใจงาม] ([68114540111
])

**Dataset:** Airline Passenger Satisfaction  
**แหล่งที่มา:** Kaggle — teejmahal20/airline-passenger-satisfaction  
**URL:** https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction

**Problem Statement:**  
โครงงานนี้วิเคราะห์ข้อมูลความพึงพอใจของผู้โดยสารสายการบิน เพื่อศึกษาว่าปัจจัยด้านการเดินทางและบริการใดมีความสัมพันธ์กับความพึงพอใจของผู้โดยสาร และสร้างโมเดล Classification เพื่อจำแนกว่าผู้โดยสารมีความพึงพอใจหรือไม่ ข้อมูลจะถูกสำรวจ ทำความสะอาด วิเคราะห์ด้วย Linear Algebra และ Statistical Learning ก่อนนำไปเปรียบเทียบโมเดลด้วย Cross-Validation

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    mean_squared_error
)

np.random.seed(42)
sns.set_theme(style="whitegrid")
print("Ready.")

---
## Part 1: Dataset Overview และ EDA เบื้องต้น

เป้าหมายคือทำความเข้าใจโครงสร้างข้อมูล ตรวจสอบ missing values และดูการกระจายของ target `satisfaction`

In [ ]:
# ─── โหลด Dataset จาก Kaggle ───────────────────────────────────────
# ใช้ Kaggle API บน Google Colab

!pip -q install kaggle

from google.colab import files
import os

print("ถ้ามีไฟล์ kaggle.json ให้กดเลือกไฟล์จากเครื่อง")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
if "kaggle.json" in uploaded:
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d teejmahal20/airline-passenger-satisfaction -p /content/airline_data

with zipfile.ZipFile("/content/airline_data/airline-passenger-satisfaction.zip", "r") as z:
    z.extractall("/content/airline_data")

print(os.listdir("/content/airline_data"))

In [ ]:
# ─── อ่าน CSV ──────────────────────────────────────────────────────
csv_files = [
    os.path.join("/content/airline_data", f)
    for f in os.listdir("/content/airline_data")
    if f.lower().endswith(".csv")
]

print("CSV files:", csv_files)

# Dataset นี้มักมี train.csv และ test.csv
if any("train" in os.path.basename(f).lower() for f in csv_files):
    train_path = [f for f in csv_files if "train" in os.path.basename(f).lower()][0]
    df = pd.read_csv(train_path)
else:
    df = pd.read_csv(csv_files[0])

# ทำความสะอาดชื่อคอลัมน์
df.columns = [c.strip() for c in df.columns]

# ลบคอลัมน์ ID ที่ไม่ใช้ในการวิเคราะห์
for col in ["Unnamed: 0", "id"]:
    if col in df.columns:
        df = df.drop(columns=col)

print(f"Shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
df.head()

In [ ]:
# ─── Descriptive Statistics + Visualization ────────────────────────
display(df.describe(include="all").T)

# Distribution ของ target
plt.figure(figsize=(7,5))
sns.countplot(data=df, x="satisfaction")
plt.title("Passenger Satisfaction Distribution")
plt.xlabel("Satisfaction")
plt.ylabel("Number of Passengers")
plt.xticks(rotation=15)
plt.show()

# Correlation heatmap ของ numeric features
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(14,10))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

---
## Part 2: CLO1 — Linear Algebra Analysis

### Task A — PCA

ใช้ PCA เพื่อลดจำนวนมิติของตัวแปรตัวเลขและดูว่าองค์ประกอบหลักสามารถอธิบายความแปรปรวนของข้อมูลได้มากเพียงใด

In [ ]:
# ─── CLO1: PCA ────────────────────────────────────────────────────
# เลือก numeric features และแทน missing values ด้วย median
X_num = df.select_dtypes(include=np.number).copy()
X_num = X_num.fillna(X_num.median())

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_num)

pca_full = PCA()
X_pca = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("Number of numeric features:", X_num.shape[1])
print("Variance explained by first 5 PCs:", explained[:5])
print("Cumulative variance of first 5 PCs:", cumulative[:5])

# Scree Plot
plt.figure(figsize=(9,5))
plt.plot(range(1, len(explained)+1), explained, marker="o")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Scree Plot")
plt.show()

# 2D PCA visualization
plt.figure(figsize=(9,6))
target_codes = df["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
}).fillna(-1)

plt.scatter(
    X_pca[:,0], X_pca[:,1],
    c=target_codes,
    alpha=0.35
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Passengers in PCA 2D Space")
plt.show()

**Insight จาก CLO1 Analysis:**  
PCA ช่วยลดมิติของ numeric features และแสดงให้เห็นว่า feature หลายตัวสามารถรวมเป็นองค์ประกอบหลักได้ การดู explained variance ช่วยประเมินว่ากี่องค์ประกอบสามารถเก็บข้อมูลส่วนใหญ่ไว้ได้

---
## Part 3: CLO2 — Statistical Learning

Dataset นี้เหมาะกับ **Classification** เพราะ target `satisfaction` เป็นข้อมูลแบบ categorical ที่มี 2 กลุ่ม คือ `satisfied` และ `neutral or dissatisfied`

In [ ]:
# ─── CLO2: Statistical Analysis ───────────────────────────────────
# แปลง target เป็น 0/1 เพื่อใช้คำนวณความสัมพันธ์
df_analysis = df.copy()
df_analysis["target"] = df_analysis["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

# ดู distribution ของ features สำคัญ
important_features = [
    c for c in [
        "Age", "Flight Distance", "Departure Delay in Minutes",
        "Arrival Delay in Minutes", "Inflight wifi service",
        "Food and drink", "Online boarding", "Seat comfort"
    ] if c in df_analysis.columns
]

for col in important_features[:4]:
    plt.figure(figsize=(7,4))
    sns.histplot(data=df_analysis, x=col, hue="satisfaction", kde=True, element="step")
    plt.title(f"Distribution of {col}")
    plt.show()

# Correlation กับ target
numeric_analysis = df_analysis.select_dtypes(include=np.number)
target_corr = numeric_analysis.corr()["target"].drop("target").sort_values(
    key=lambda x: abs(x), ascending=False
)

print("Top features correlated with satisfaction:")
display(target_corr.head(10).to_frame("correlation"))

plt.figure(figsize=(8,5))
target_corr.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Correlations with Satisfaction")
plt.xlabel("Correlation")
plt.show()

In [ ]:
# ─── Bias-Variance: model complexity ของ KNN ──────────────────────
# ใช้ sample เพื่อลดเวลาในการคำนวณบน Colab
feature_candidates = [
    "Age", "Flight Distance", "Departure Delay in Minutes",
    "Arrival Delay in Minutes", "Inflight wifi service",
    "Food and drink", "Online boarding", "Seat comfort",
    "Inflight entertainment", "Leg room service", "Cleanliness"
]
feature_cols = [c for c in feature_candidates if c in df.columns]

X = df[feature_cols].copy().fillna(df[feature_cols].median())
y = df["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

k_values = [1, 3, 5, 7, 11, 15, 21]
train_errors = []
test_errors = []

for k in k_values:
    knn = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    knn.fit(X_train, y_train)
    train_errors.append(1 - knn.score(X_train, y_train))
    test_errors.append(1 - knn.score(X_test, y_test))

plt.figure(figsize=(8,5))
plt.plot(k_values, train_errors, marker="o", label="Train Error")
plt.plot(k_values, test_errors, marker="o", label="Test Error")
plt.xlabel("K in KNN")
plt.ylabel("Error")
plt.title("Bias-Variance: KNN Model Complexity")
plt.legend()
plt.show()

**Insight จาก CLO2 Analysis:**  
ตัวแปรด้านบริการและประสบการณ์การเดินทางหลายตัวมีความสัมพันธ์กับ `satisfaction` แต่ correlation ไม่ได้หมายความว่าเป็นเหตุเป็นผลโดยตรง กราฟ Train/Test Error ของ KNN ใช้ดูผลของ model complexity และช่วยสังเกตแนวโน้ม overfitting/underfitting

---
## Part 4: CLO3 — Model Building

เนื่องจาก `satisfaction` เป็น categorical target จึงใช้ Classification

- Model 1: Logistic Regression
- Model 2: K-Nearest Neighbors (KNN)

In [ ]:
# ─── เตรียมข้อมูลสำหรับ Classification ───────────────────────────
X = df[feature_cols].copy().fillna(df[feature_cols].median())
y = df["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# ─── CLO3: Model 1 — Logistic Regression ──────────────────────────
model1 = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000, random_state=42))
])

model1.fit(X_train, y_train)

y_train_pred1 = model1.predict(X_train)
y_test_pred1 = model1.predict(X_test)

print("Model 1: Logistic Regression")
print("Train Accuracy:", accuracy_score(y_train, y_train_pred1))
print("Test Accuracy :", accuracy_score(y_test, y_test_pred1))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred1))

cm1 = confusion_matrix(y_test, y_test_pred1)
sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# ─── CLO3: Model 2 — KNN ──────────────────────────────────────────
model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])

model2.fit(X_train, y_train)

y_train_pred2 = model2.predict(X_train)
y_test_pred2 = model2.predict(X_test)

print("Model 2: KNN")
print("Train Accuracy:", accuracy_score(y_train, y_train_pred2))
print("Test Accuracy :", accuracy_score(y_test, y_test_pred2))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred2))

cm2 = confusion_matrix(y_test, y_test_pred2)
sns.heatmap(cm2, annot=True, fmt="d", cmap="Blues")
plt.title("KNN - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**สรุป Model Comparison**

| Model | Train Accuracy | Test Accuracy | รายละเอียด |
|---|---:|---:|---|
| Logistic Regression | ดูจากผลรัน | ดูจากผลรัน | Linear classification model |
| KNN | ดูจากผลรัน | ดูจากผลรัน | ใช้ระยะห่างของข้อมูลในการจำแนก |

**Insight:** เปรียบเทียบ Train/Test Accuracy และ Classification Report เพื่อดูประสิทธิภาพและความแตกต่างระหว่างโมเดล

---
## Part 5: CLO4 — Model Selection

ใช้ 5-Fold Cross-Validation เพื่อเปรียบเทียบอย่างน้อย 3 models โดยใช้ Accuracy เป็น metric เนื่องจากเป็น Classification

In [ ]:
# ─── CLO4: 5-Fold Cross-Validation ────────────────────────────────
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    )
}

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy", n_jobs=-1)
    cv_results[name] = scores.mean()
    print(f"{name}: Mean CV Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

plt.figure(figsize=(9,5))
plt.bar(cv_results.keys(), cv_results.values())
plt.ylabel("Mean 5-Fold CV Accuracy")
plt.title("Model Comparison using 5-Fold Cross-Validation")
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.show()

best_model_name = max(cv_results, key=cv_results.get)
print("Best Model from Cross-Validation:", best_model_name)

**Best Model จาก Cross-Validation:** ให้ใช้ค่าที่ได้จากตัวแปร `best_model_name`

**เหตุผล:** เลือกโมเดลจากค่า Mean 5-Fold CV Accuracy ซึ่งเป็นค่าเฉลี่ยจากหลายชุด train/validation จึงช่วยลดผลกระทบจากการแบ่งข้อมูลเพียงครั้งเดียว

---
## Part 6: สรุปผลและ Data Story

### 6.1 สรุปผลการวิเคราะห์
- **CLO1:** ใช้ PCA ลดมิติของ numeric features และดู explained variance
- **CLO2:** วิเคราะห์ distribution และ correlation ของ features กับ satisfaction รวมถึงดู Bias-Variance ด้วย KNN
- **CLO3:** สร้าง Logistic Regression และ KNN สำหรับ Classification
- **CLO4:** ใช้ 5-Fold Cross-Validation เปรียบเทียบ Logistic Regression, KNN และ Random Forest

### 6.2 Limitations
- ข้อมูลเป็นข้อมูลจาก dataset ที่จัดเตรียมไว้แล้ว จึงอาจไม่ครอบคลุมผู้โดยสารทุกสายการบินหรือทุกช่วงเวลา
- ตัวแปรบางตัวเป็นคะแนนบริการ ซึ่งเป็นข้อมูลเชิงอันดับและอาจมีข้อจำกัดเมื่อนำไปวิเคราะห์แบบ numeric โดยตรง
- ผลลัพธ์ของโมเดลขึ้นอยู่กับ features ที่เลือกและการ preprocessing

### 6.3 Future Work
- ทดลอง Feature Selection และ Hyperparameter Tuning
- ทดลองโมเดล Classification เพิ่มเติม
- วิเคราะห์แต่ละสายการบินหรือกลุ่มผู้โดยสารแยกกัน
- สร้าง Dashboard เพื่อสื่อสารผลการวิเคราะห์

### 6.4 Connection กับ Topics ในวิชา
- Linear Algebra → Standardization และ PCA
- Statistical Learning → Correlation และ Bias-Variance
- Classification → Logistic Regression และ KNN
- Model Selection → 5-Fold Cross-Validation

---
**Acknowledgements:** Kaggle Dataset — Airline Passenger Satisfaction และ Google Colab / Python / scikit-learn